# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides users through loading, exploring, and processing the FAIR⁲ dataset described by a Croissant schema, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}\n")
print(f"Published: {metadata.date_published}\n")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`. All entities are referenced by their `@id` fields as per best practices.

In [ ]:
# Display available record sets and their fields by @id
print("Available record sets in metadata.record_sets:")
record_set_ids = []
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else ''}")
    record_set_ids.append(rs.id)
    # List the fields for each record set
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    - Field @id: {field.id}, name: {field.name}, data type: {field.data_type}")
    print()
if not record_set_ids:
    print("No record sets found in the metadata. Check the dataset definition or metadata attributes.")    

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Try extracting from all available record sets, if they exist.
dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        try:
            # Load all records from this record set
            records = list(dataset.records(record_set=record_set_id))
            if records:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
                print(f"Columns: {df.columns.tolist()}")
                display(df.head())
            else:
                print(f"No records found for RecordSet @id: {record_set_id}")
        except Exception as e:
            print(f"Error loading records for {record_set_id}: {e}")
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply standard data operations prepared for EDA.
*Instructions:*
- Select a numeric field by its field `@id` for analysis (refer to the printed fields above).
- Filter, normalize, and group data as required.

In [ ]:
# Example setup for EDA if a record set is available
# Replace these IDs as per your real field IDs discovered above (shown in previous output)

# Choose the first record set and a numeric field for demonstration
import numpy as np
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    df = dataframes[chosen_record_set_id]

    # Try to pick a numeric column automatically
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric columns detected. Please manually select a numeric field.")
        numeric_field_id = None

    # Proceed if a numeric field is available
    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to find a group-able column (object/string, with reasonable cardinality)
        non_numeric_cols = df.select_dtypes(include='object').columns.tolist()
        group_field = None
        for col in non_numeric_cols:
            unique_vals = df[col].nunique()
            if 1 < unique_vals < 10:
                group_field = col
                print(f"Using group field: {group_field}")
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable group-by field found.")
    else:
        print("No numeric field selected for EDA.")
else:
    print("No dataframe loaded for EDA.")

## 5. Visualization
Visualize distributions and relationships in the dataset. Choose relevant fields with their `@id` (column names as loaded) for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If a group field, show boxplot
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
- This notebook demonstrated the use of the `mlcroissant` library and Croissant schema to explore the FAIR² dataset using direct schema entity references (`@id`).
- Users can adapt field and record set IDs in the code above to analyze specific aspects further as needed.

Further analysis can be performed with domain-appropriate transformations, modeling, or visualizations on the loaded DataFrames.